[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/00-setup/notebooks/01-environment-check.ipynb)

# 00 · Notebook 1 — Environment check

**Goal:** confirm your Python setup works, and learn what a notebook really is.

You will:
1. Run your first cells and see how outputs appear.
2. Find out which Python is running and whether you are in a **virtual environment** (a private folder of packages for one project).
3. List installed **packages** (code other people wrote that you can install).
4. Look inside a `.ipynb` file and see that it is plain **JSON** (a text format of `{ "key": value }` pairs).
5. See why the order you run cells in matters.

How to run a cell: click inside it and press **Shift + Enter**. This works in Jupyter, VS Code and Google Colab.

## 1. Your first cell

A **code cell** holds Python. When you run it, the result appears under it. The last line of a cell is shown automatically, even without `print`.

In [1]:
print("Hello from a notebook!")
2 + 2

Hello from a notebook!


4

## 2. Which Python is this?

The `sys` module (a **module** is a file of ready-made Python code) knows about the Python that is running this notebook.
- `sys.version` is the version number.
- `sys.executable` is the exact program on disk. If it contains `.venv`, you are using a virtual environment.

In [2]:
import sys
import platform

print("Python version :", sys.version.split()[0])
print("Python program :", sys.executable)
print("Operating sys. :", platform.system(), platform.release())

if sys.version_info < (3, 10):
    print("WARNING: this course needs Python 3.10 or newer.")
else:
    print("Version OK for this course.")

Python version : 3.13.18
Python program : /usr/local/bin/python3
Operating sys. : Linux 6.18.44-fc-v726
Version OK for this course.


## 3. Am I inside a virtual environment? Am I in Colab?

Inside a virtual environment, `sys.prefix` (where packages are installed) differs from `sys.base_prefix` (the original Python install).

Google Colab is detected by trying to import its special `google.colab` module. `try` / `except` means "try this; if it fails with this error, do that instead" — you will learn it properly in module 01.

In [3]:
in_venv = sys.prefix != sys.base_prefix
print("Inside a virtual environment:", in_venv)

try:
    import google.colab  # this module only exists on Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Google Colab:", IN_COLAB)

Inside a virtual environment: False
Running in Google Colab: False


## 4. Which packages are installed?

`importlib.metadata` is part of Python's **standard library** (the modules that come with Python). It can tell us each package's version. We check the packages this course relies on. "missing" is not an error — you install those later with `pip`.

In [4]:
import importlib.metadata as meta

course_packages = ["pip", "python-dotenv", "ipykernel", "numpy", "pandas",
                   "matplotlib", "scikit-learn", "torch", "transformers"]

for name in course_packages:
    try:
        print(f"[ok]      {name:<15} {meta.version(name)}")
    except meta.PackageNotFoundError:
        print(f"[missing] {name:<15} install with: python -m pip install {name}")

all_names = {d.metadata["Name"] for d in meta.distributions() if d.metadata["Name"]}
print(f"\nTotal packages installed: {len(all_names)}")

[ok]      pip             26.2.1.post1
[ok]      python-dotenv   1.2.3
[ok]      ipykernel       7.4.0
[ok]      numpy           2.5.3
[ok]      pandas          3.0.5
[ok]      matplotlib      3.11.2
[ok]      scikit-learn    1.9.1
[ok]      torch           2.15.0+cpu
[ok]      transformers    5.23.0

Total packages installed: 286


## 5. Running terminal commands from a notebook

A line starting with `!` is sent to the **terminal** (the text window for typing commands) instead of Python. This is handy in Colab, where you have no separate terminal. Here we ask the terminal for the Python version and pip's version.

In [5]:
!python --version
!python -m pip --version

Python 3.13.18


pip 26.2.1 from /usr/local/lib/python3.13/site-packages/pip (python 3.13)


## 6. Run the module's checker script

The module ships `code/check_env.py`, a full health report. It prints whether API keys are set **without printing the keys**.

The cell below finds the script next to this notebook. In Colab the repo files are not there, so it downloads the script from GitHub first.

In [6]:
from pathlib import Path
import subprocess, urllib.request

script = Path("../code/check_env.py")
if not script.exists():
    # Running in Colab or elsewhere: fetch the script from the course repo.
    url = "https://raw.githubusercontent.com/vvvvvivekkk/AI-Engineer/main/00-setup/code/check_env.py"
    script = Path("check_env.py")
    try:
        urllib.request.urlretrieve(url, script)
    except Exception as err:
        print("Could not download the script:", err)

if script.exists():
    result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
    print(result.stdout[-3000:])

AI Engineer course - environment check

1. Python
---------
Version     : 3.13.18
Executable  : /usr/local/bin/python3
OS          : Linux 6.18.44-fc-v726
Status      : OK

2. Virtual environment
----------------------
Active venv : none (packages go into your main Python install)
Tip         : create one with  python -m venv .venv  and activate it

3. Packages
-----------
  [ok]      pip             26.2.1.post1 installs other packages
  [ok]      python-dotenv   1.2.3        loads API keys from a .env file
  [missing] jupyterlab      -            local notebooks in the browser
  [missing] notebook        -            classic Jupyter notebook interface
  [ok]      ipykernel       7.4.0        lets Jupyter / VS Code run Python cells
  [ok]      numpy           2.5.3        fast number arrays (module 02)
  [ok]      pandas          3.0.5        tables of data (module 02)
  [ok]      matplotlib      3.11.2       charts (module 02)
  [ok]      scikit-learn    1.9.1        classic machine 

## 7. A notebook is JSON

A `.ipynb` file is a text file in JSON format. It stores every cell **and its outputs**. That is why you must never print a secret in a notebook: the printed text gets saved in the file.

The `nbformat` package (installed with Jupyter) can build a notebook in memory. We make a tiny two-cell notebook and print its JSON text.

In [7]:
import json
import nbformat
from nbformat.v4 import new_notebook, new_markdown_cell, new_code_cell

tiny = new_notebook()
tiny.cells.append(new_markdown_cell("# My tiny notebook"))
tiny.cells.append(new_code_cell("print(2 + 2)"))

text = nbformat.writes(tiny)          # the exact text that would be saved to a .ipynb file
print(text[:700])

{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "3261e5a9",
   "metadata": {},
   "source": [
    "# My tiny notebook"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "0478b1f4",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(2 + 2)"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}


Look at the printout: a `"cells"` list, each cell with a `"cell_type"` (`markdown` or `code`) and a `"source"` (its text). Code cells also have `"outputs"` and an `"execution_count"` (the number in `[ ]` beside a cell).

Now we read **this** notebook file, if it is available, and count its cells by type.

In [8]:
nb_file = Path("01-environment-check.ipynb")
if nb_file.exists():
    data = json.loads(nb_file.read_text(encoding="utf-8"))
    kinds = [c["cell_type"] for c in data["cells"]]
    print("Cells in this notebook:", len(kinds))
    print("Markdown cells:", kinds.count("markdown"), "| Code cells:", kinds.count("code"))
else:
    print("This notebook's file is not on disk here (normal in Colab).")

Cells in this notebook: 22
Markdown cells: 12 | Code cells: 10


## 8. Run order matters: the kernel's memory

The **kernel** is the Python process behind the notebook. It remembers variables in the order you *ran* cells, not the order they appear.

Try this experiment:
1. Run the next two cells in order. You will see `10`.
2. Change `price = 5` to `price = 7` in the first cell but **do not run it**. Run only the second cell again. It still prints `10` — the kernel never saw your edit.
3. Fix: **restart the kernel and run all** (Jupyter: *Kernel → Restart Kernel and Run All Cells…*; Colab: *Runtime → Restart session and run all*; VS Code: *Restart*, then *Run All*).

In [9]:
price = 5
quantity = 2

The cell below uses the variables created above.

In [10]:
total = price * quantity
print(total)

10


## Summary

- You ran code cells and saw outputs.
- You found which Python is running and whether a virtual environment is active.
- You listed installed packages and ran `check_env.py`.
- You saw that a `.ipynb` file is JSON that also stores outputs — so never print secrets.
- When a notebook behaves strangely: restart the kernel and run all.

Next: **Notebook 2 — working with `.env` safely**.